# Evaluate the models

Evaluates last pth of each run with the config it was trained with

- Every test point is predicted by 4 overlapping blocks, then the softmax outputs are summed.
- mIoU averages classes present in the test tile
- PAvPU uses K = 30 nearest neighbours in metric coordinates, the threshold (median patch uncertainty) is cached per dataset, so it is identical for every run

## Setup

In [ ]:
import os, sys
ROOT = os.path.abspath('..')
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 50)

from src import paths
print('project folder:', paths.PROJECT_ROOT)

import json
import torch
from src.infer import evaluateRun, loadRunConfig
from src.prepare import loadManifest

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
DATASET        = 'paris_lille'
VARIANTS       = ['baseline', 'uncertainty']
SEEDS          = [42, 43, 44]
EVAL_OVERRIDES = ['evaluation.inferBatchBlocks=16']

## Evaluate

In [ ]:
results = {}
for seed in SEEDS:
    for variant in VARIANTS:
        print(f'\n===== {DATASET} / {variant} / seed {seed} =====')
        cfg = loadRunConfig(DATASET, variant, seed, EVAL_OVERRIDES)
        results[(variant, seed)] = evaluateRun(cfg, variant, seed, device)
        torch.cuda.empty_cache()

## Summary

In [ ]:
summary = pd.DataFrame([{'variant': v, 'seed': s, 'OA_all': m['OA_all'], 'mIoU': m['mIoU'], 'PAvPU': m['PAvPU']}
                        for (v, s), m in results.items()]).sort_values(['variant', 'seed'])
display(summary.round(4))
display(summary.groupby('variant')[['OA_all', 'mIoU', 'PAvPU']].agg(['mean', 'std']).round(4))

## Per-class IoU

In [ ]:
manifest = loadManifest(cfg)
idxToRaw = {v: k for k, v in manifest['labelMap'].items()}
perClass = pd.DataFrame({f'{v} s{s}': m['classIoU'] for (v, s), m in results.items()})
perClass.index = [f"{i} (raw {idxToRaw.get(i, '?')})" for i in range(len(perClass))]
perClass['in_mIoU'] = [i in next(iter(results.values()))['classesAveraged'] for i in range(len(perClass))]
display(perClass.round(4))

## PAvPU breakdown

In [ ]:
rows = []
for (v, s) in results:
    with open(os.path.join(paths.runDir(cfg, v, s), 'pavpu.json')) as f:
        p = json.load(f)
    rows.append({'variant': v, 'seed': s, **{k: p[k] for k in ('AC_pct', 'AU_pct', 'IC_pct', 'IU_pct', 'PAvPU',
                 'P_uncertain_given_wrong', 'P_correct_given_certain', 'uncertaintyThreshold')}})
display(pd.DataFrame(rows).round(4))